# 03: Adversarial Defenses & Robust Optimization
**Adversarial Training, Defensive Distillation, Randomized Smoothing & Preprocessing**

---

## 1. The Min-Max Robust Optimization Problem
Adversarial training is mathematically formulated as a saddle-point (min-max) game:
$$\min_{oldsymbol{	heta}} \mathbb{E}_{(\mathbf{x}, y) \sim \mathcal{D}} \left[ \max_{oldsymbol{\delta} \in \mathcal{S}} \mathcal{L}(oldsymbol{	heta}, \mathbf{x} + oldsymbol{\delta}, y) ight]$$
- **Inner Maximization:** Finds the strongest adversarial perturbation $oldsymbol{\delta}$ (e.g. via FGSM or PGD).
- **Outer Minimization:** Updates network parameters $oldsymbol{	heta}$ to minimize the adversarial loss.


In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

from adv_studio.models import get_model
from adv_studio.data import get_mnist_loaders
from adv_studio.attacks import FGSMAttack, PGDAttack
from adv_studio.defenses import BitDepthReduction, SpatialSmoothing, TotalVariationDenoising
from adv_studio.evaluation import compute_robust_accuracy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, test_loader = get_mnist_loaders(data_dir="../DATA", batch_size=64)

clean_model = get_model("simple_cnn", pretrained_path="../checkpoints/mnist_cnn.pth", device=device)
adv_model = get_model("simple_cnn", pretrained_path="../checkpoints/mnist_cnn_adv_trained.pth", device=device)

clean_model.eval()
adv_model.eval()


## 2. Comparing Clean-Trained vs Adversarially-Trained Model


In [ ]:
epsilons = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
clean_accs = []
adv_accs = []

for eps in epsilons:
    if eps == 0.0:
        c_res = compute_robust_accuracy(clean_model, test_loader, device=device, max_batches=10)
        a_res = compute_robust_accuracy(adv_model, test_loader, device=device, max_batches=10)
        clean_accs.append(c_res['clean_accuracy'])
        adv_accs.append(a_res['clean_accuracy'])
    else:
        atk_clean = FGSMAttack(clean_model, epsilon=eps, device=device)
        atk_adv = FGSMAttack(adv_model, epsilon=eps, device=device)
        c_res = compute_robust_accuracy(clean_model, test_loader, attack=atk_clean, device=device, max_batches=10)
        a_res = compute_robust_accuracy(adv_model, test_loader, attack=atk_adv, device=device, max_batches=10)
        clean_accs.append(c_res['robust_accuracy'])
        adv_accs.append(a_res['robust_accuracy'])

plt.figure(figsize=(7, 4.5))
plt.plot(epsilons, [a*100 for a in clean_accs], marker="o", label="Clean-Trained Model", color="#ef4444")
plt.plot(epsilons, [a*100 for a in adv_accs], marker="s", label="Adversarially-Trained Model", color="#10b981")
plt.title("Robustness Comparison: Clean vs Adversarial Training")
plt.xlabel("Epsilon")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


## 3. Input Preprocessing Defense Layers (Bit-Depth, Spatial Blur, TV)


In [ ]:
attack = FGSMAttack(clean_model, epsilon=0.2, device=device)
defenses = {
    "No Defense": None,
    "Bit-Depth (3-bit)": BitDepthReduction(step=3),
    "Spatial Blur (sigma=1.0)": SpatialSmoothing(sigma=1.0),
    "TV Denoising": TotalVariationDenoising(weight=0.05),
}

for name, def_layer in defenses.items():
    res = compute_robust_accuracy(clean_model, test_loader, attack=attack, defense=def_layer, device=device, max_batches=10)
    print(f"Defense: {name:<25} | Robust Acc: {res['robust_accuracy']*100:.2f}%")
